Find the best-selling item for each month (no need to separate months by year). The best-selling item is determined by the highest total sales amount, calculated as: total_paid = unitprice * quantity. A negative quantity indicates a return or cancellation (the invoice number begins with 'C'. To calculate sales, ignore returns and cancellations. Output the month, description of the item, and the total amount paid.

In [0]:
%sql
WITH filtered_sales AS (
    SELECT 
        MONTH(invoice_date) AS month,
        description,
        unitprice * quantity AS total_paid
    FROM online_retail
    WHERE quantity > 0
      AND invoice_no NOT LIKE 'C%'
),
agg_sales AS (
    SELECT 
        month,
        description,
        SUM(total_paid) AS total_sales
    FROM filtered_sales
    GROUP BY month, description
),
ranked AS (
    SELECT 
        month,
        description,
        total_sales,
        RANK() OVER (PARTITION BY month ORDER BY total_sales DESC) AS rnk
    FROM agg_sales
)
SELECT 
    month,
    description,
    total_sales
FROM ranked
WHERE rnk = 1;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Step 1: Filter valid sales
df_filtered = df.filter(
    (F.col("quantity") > 0) &
    (~F.col("invoice_no").startswith("C"))
)

# Step 2: Extract month
df_filtered = df_filtered.withColumn("month", F.month("invoice_date"))

# Step 3: Compute total_paid
df_filtered = df_filtered.withColumn(
    "total_paid", F.col("unitprice") * F.col("quantity")
)

# Step 4: Aggregate per month + item
agg_df = df_filtered.groupBy("month", "description").agg(
    F.sum("total_paid").alias("total_sales")
)

# Step 5: Rank within month
window_spec = Window.partitionBy("month").orderBy(F.col("total_sales").desc())

ranked_df = agg_df.withColumn("rnk", F.rank().over(window_spec))

# Step 6: Get best-selling items
result = ranked_df.filter(F.col("rnk") == 1)

result.show()